In [ ]:
!pip install -U accelerate peft
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q Pillow==10.4.0
!pip install -U transformers==4.57.0
!pip install trl  # for SFTTrainer
!pip install -U torch
!pip install clearml

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [peft]1/2 [peft]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, but you have peft 0.18.1 which is incom

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [5]:
import os
import dotenv 
dotenv.load_dotenv()

from clearml import Task, Dataset

import pandas as pd
import numpy as np

In [ ]:

Task.set_credentials(
     api_host="https://api.clear.ml",
     web_host="https://app.clear.ml",
     files_host="https://files.clear.ml",
     key=os.getenv('clear_ml_key'), # get it from https://app.clear.ml/settings/workspace-configuration
     secret=os.getenv('clear_ml_secret') # get it from https://app.clear.ml/settings/workspace-configuration
)

In [7]:
# Utility functions

def parse_basic(path):
    """""
    Note: The structure of the metadata files is as follows:
    dish_id, total_calories, total_mass, total_fat, total_carb, total_protein, ingr_1_id, ingr_1_name, ingr_1_grams, ingr_1_calories, ingr_1_fat, ingr_1_carb, ingr_1_protein, ingr_2_id, ingr_2_name, ...

    Some dishes have more than 5 ingredients, while some have fewer.
    """""
    # parse_basic, we only care about the first 6 columns for now (dish_id, total_calories, total_mass, total_fat, total_carb, total_protein)
    rows = []

    with open(path) as f:
        for line in f:
            parts = line.strip().split(",")

            # เอาแค่ 6 column แรก
            try:
                rows.append({
                    "dish_id": parts[0].replace("dish_", ""),
                    "total_calories": float(parts[1]),
                    "total_mass": float(parts[2]),
                    "total_fat": float(parts[3]),
                    "total_carb": float(parts[4]),
                    "total_protein": float(parts[5]),
                })
            except:
                continue

    return pd.DataFrame(rows)

In [8]:
ds = Dataset.get(
    dataset_name="nutrition5k_dataset",
    dataset_project="NutritionAnalyser"
)

local_path = ds.get_local_copy()
print("Dataset path:", local_path)

# load both metadata files and concatenate them into a single dataframe
df1 = parse_basic(os.path.join(local_path, "dish_metadata_cafe1.csv"))
df2 = parse_basic(os.path.join(local_path, "dish_metadata_cafe2.csv"))

df = pd.concat([df1, df2], ignore_index=True)

print(df.shape)
df.head()

Dataset path: /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_b6323a4330fa49a8b2a210b61ed415fc
(5006, 6)


,dish_id,total_calories,total_mass,total_fat,total_carb,total_protein
0,1561662216,300.794281,193.0,12.387489,28.218290,18.633970
1,1562688426,137.569992,88.0,8.256000,5.190000,10.297000
2,1561662054,419.438782,292.0,23.838249,26.351543,25.910593
3,1562008979,382.936646,290.0,22.224644,10.173570,35.345387
4,1560455030,20.590000,103.0,0.148000,4.625000,0.956000


#### Map image path

In [9]:
# add image path column
df["image_path"] = df["dish_id"].apply(
    lambda x: os.path.join(local_path, f"dish_{x}_rgb.png")
)

# filter only rows where image exists
df = df[df["image_path"].apply(os.path.exists)] 
print("Final samples:", len(df))

Final samples: 3079


### Feature engineering

In [10]:
# remove rows with zero calories (these are likely errors in the data and will cause issues with log-transforming the target variable)
df_nozero = df[df["total_calories"] > 0]

# log-transform nutritional values to reduce skewness in the distribution and make it easier for the model to learn
df_nozero["calories_log"] = np.log1p(df_nozero["total_calories"])
df_nozero["protein_log"] = np.log1p(df_nozero["total_protein"])
df_nozero["carb_log"] = np.log1p(df_nozero["total_carb"])
df_nozero["fat_log"] = np.log1p(df_nozero["total_fat"])

/tmp/ipykernel_434/2491069597.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nozero["calories_log"] = np.log1p(df_nozero["total_calories"])
/tmp/ipykernel_434/2491069597.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nozero["protein_log"] = np.log1p(df_nozero["total_protein"])
/tmp/ipykernel_434/2491069597.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentat

In [ ]:
from sklearn.model_selection import train_test_split

df_nozero["cal_bin"] = pd.qcut(df_nozero["total_calories"], q=5, labels=False) # create 5 bins of calories for stratified splitting


# Split into train (60%), val (20%), test (20%)
# stratify by calorie bins to ensure similar distribution of calories in train and test sets
train_df, temp_df = train_test_split(
    df_nozero, 
    test_size=0.4, 
    random_state=42, 
    stratify=df_nozero["cal_bin"]
)
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df["cal_bin"]
)


train_df = train_df.drop(columns=["cal_bin"]).reset_index(drop=True)
val_df = val_df.drop(columns=["cal_bin"]).reset_index(drop=True)
test_df = test_df.drop(columns=["cal_bin"]).reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

# select only the columns we need for training the model
train_df = train_df[[
    "image_path",
    "calories_log",
    "protein_log",
    "carb_log",
    "fat_log"
]]

val_df = val_df[[
    "image_path",
    "calories_log",
    "protein_log",
    "carb_log",
    "fat_log"
]]

test_df = test_df[[
    "image_path",
    "calories_log",
    "protein_log",
    "carb_log",
    "fat_log"
]]

Train: 2025
Test: 868


/tmp/ipykernel_434/1179909298.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nozero["cal_bin"] = pd.qcut(df_nozero["total_calories"], q=5, labels=False) # create 5 bins of calories for stratified splitting


#### VLM Fine-Tuning 


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import json

PROMPT = """You are a food nutrition analyzer. Analyze the food image and return ONLY a JSON object with this exact schema:

{
  "nutritional_summary": {
    "calories_kcal": <number>,
    "protein_g": <number>,
    "carbohydrate_g": <number>,
    "fat_g": <number>
  }
}

Rules:
- Return JSON only. No explanation, no markdown, no code blocks.
- All values must be numbers, not strings.
- Do not add extra fields."""

class FoodVLMDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image.thumbnail((512, 512), Image.Resampling.LANCZOS)

        # Build target JSON from labels (targets were stored as log1p; inverse is expm1, not exp)
        import math

        target = {
            "nutritional_summary": {
                "calories_kcal": round(math.expm1(row["calories_log"])),
                "protein_g": round(math.expm1(row["protein_log"]), 2),
                "carbohydrate_g": round(math.expm1(row["carb_log"]), 2),
                "fat_g": round(math.expm1(row["fat_log"]), 2),
            }
        }

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPT}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": json.dumps(target)}]
            }
        ]
        return {"messages": messages, "image": image}

#### Load Model + LoRA

In [ ]:
import os
from pathlib import Path

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import PeftModel, LoraConfig, get_peft_model
import torch

# Directory for best adapter + training_state.pt (resume). Override with env FINETUNE_CHECKPOINT_DIR.
CHECKPOINT_DIR = Path(os.environ.get("FINETUNE_CHECKPOINT_DIR", "./food-vlm-finetuned-best"))

base_model_name = "unsloth/Qwen3-VL-2B-Instruct-bnb-4bit"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(base_model_name, trust_remote_code=True)

# Check processor's expected image size (optional but recommended)
print("Processor image size:", processor.image_processor.size)

# Load existing LoRA from Ateeqq/food-analysis, then add new LoRA on top
model = PeftModel.from_pretrained(model, "Ateeqq/food-analysis", is_trainable=False)
model = model.merge_and_unload()  # merge the existing adapter

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Resume YOUR LoRA if this notebook already saved an adapter here; otherwise start a new adapter.
_adapter_files_ok = (
    CHECKPOINT_DIR.is_dir()
    and (CHECKPOINT_DIR / "adapter_config.json").exists()
    and (
        (CHECKPOINT_DIR / "adapter_model.safetensors").exists()
        or (CHECKPOINT_DIR / "adapter_model.bin").exists()
    )
)
if _adapter_files_ok:
    print(f"Resuming trainable LoRA from {CHECKPOINT_DIR.resolve()}")
    model = PeftModel.from_pretrained(model, str(CHECKPOINT_DIR), is_trainable=True)
    ADAPTER_RESUMED = True
else:
    print("Initializing a new LoRA adapter (no prior checkpoint in CHECKPOINT_DIR)")
    model = get_peft_model(model, lora_config)
    ADAPTER_RESUMED = False

model.print_trainable_parameters()

2026-03-28 14:34:58.758827: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 14:34:58.940503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774708498.968044     434 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774708498.981153     434 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-28 14:34:59.185494: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

trainable params: 20,512,768 || all params: 2,148,044,800 || trainable%: 0.9550


#### Collator + Training Loop

In [ ]:
from torch.utils.data import DataLoader
import torch

# Longer context reduces accidental truncation of multimodal + JSON completions
_MAX_SEQ_LEN = 2048


def collate_fn(batch):
    """Batch samples for the vision-language model and build masked labels for SFT.

    Each sample is expected to provide ``messages`` (chat turns) and ``image`` as
    produced by ``FoodVLMDataset``. The processor tokenizes the full conversation
    (user image + prompt + assistant JSON) and a **user-only** prefix with
    ``add_generation_prompt=True`` so its token IDs align with the start of the
    full sequence.

    Labels are ``input_ids`` with ``-100`` on positions that must not contribute to
    loss: the user/prefix span (including image tokens), padding, and optionally a
    fallback to the full sequence for a row if masking would leave no supervised
    tokens (e.g. aggressive truncation).

    Args:
        batch: List of dicts with keys ``messages`` and ``image``.

    Returns:
        A dict of tensors suitable for ``model(**batch)``, including ``labels``.
    """
    images = [item["image"] for item in batch]
    texts = []
    prefix_texts = []
    for item in batch:
        texts.append(
            processor.apply_chat_template(
                item["messages"], tokenize=False, add_generation_prompt=False
            )
        )
        user_only = item["messages"][:1]
        prefix_texts.append(
            processor.apply_chat_template(
                user_only, tokenize=False, add_generation_prompt=True
            )
        )

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=_MAX_SEQ_LEN,
    )

    prefix_inputs = processor(
        text=prefix_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=_MAX_SEQ_LEN,
    )

    labels = inputs["input_ids"].clone()
    bsz = labels.size(0)
    for i in range(bsz):
        plen = int(prefix_inputs["attention_mask"][i].sum().item())
        flen = int(inputs["attention_mask"][i].sum().item())
        p_ids = prefix_inputs["input_ids"][i, :plen]
        f_ids = inputs["input_ids"][i, :flen]
        if plen > 0 and flen >= plen and torch.equal(p_ids, f_ids[:plen]):
            assistant_start = plen
        else:
            # Truncation / template mismatch: use longest matching prefix
            assistant_start = 0
            for j in range(min(plen, flen)):
                if prefix_inputs["input_ids"][i, j] != inputs["input_ids"][i, j]:
                    assistant_start = j
                    break
            else:
                assistant_start = min(plen, flen)
        labels[i, :assistant_start] = -100

    labels[inputs["attention_mask"] == 0] = -100
    for i in range(bsz):
        if (labels[i] != -100).sum().item() == 0:
            labels[i] = inputs["input_ids"][i].clone()
            labels[i][inputs["attention_mask"][i] == 0] = -100
    inputs["labels"] = labels
    return inputs


train_dataset = FoodVLMDataset(train_df, processor)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

val_dataset = FoodVLMDataset(val_df, processor)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

test_dataset = FoodVLMDataset(test_df, processor)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

In [ ]:
from transformers import get_scheduler
from torch.optim import AdamW
from tqdm import tqdm
import torch


def _model_device(mod):
    for p in mod.parameters():
        return p.device
    return torch.device("cpu")


def _optimizer_state_to_device(opt, dev):
    for st in opt.state.values():
        for k, v in list(st.items()):
            if torch.is_tensor(v):
                st[k] = v.to(dev)


optimizer = AdamW(model.parameters(), lr=1e-4)
num_epochs = 5
device = _model_device(model)

scheduler = get_scheduler(
    "cosine",
    optimizer,
    num_warmup_steps=10,
    num_training_steps=len(train_loader) * num_epochs,
)

# Resume epoch / optimizer / scheduler / best val when we loaded an adapter and training_state.pt exists
training_state_path = CHECKPOINT_DIR / "training_state.pt"
start_epoch = 0
best_val_loss = float("inf")
state = None
if ADAPTER_RESUMED and training_state_path.is_file():
    state = torch.load(training_state_path, map_location="cpu", weights_only=False)
    start_epoch = int(state.get("next_epoch", 0))
    best_val_loss = float(state.get("best_val_loss", float("inf")))
    optimizer.load_state_dict(state["optimizer"])
    _optimizer_state_to_device(optimizer, device)
    sched_ok = (
        state.get("num_epochs") == num_epochs
        and state.get("len_train_loader") == len(train_loader)
    )
    if sched_ok:
        scheduler.load_state_dict(state["scheduler"])
    else:
        print(
            "Warning: num_epochs or len(train_loader) changed vs checkpoint; "
            "optimizer restored but scheduler reset to a fresh cosine schedule."
        )
    print(
        f"Loaded training_state.pt — next loop epoch index {start_epoch} "
        f"(run epochs {start_epoch + 1}..{num_epochs} in 1-based logs), best_val_loss={best_val_loss:.4f}"
    )
elif training_state_path.is_file() and not ADAPTER_RESUMED:
    print(
        "Note: training_state.pt exists but no adapter was resumed; starting epoch 0 with a fresh optimizer."
    )

if start_epoch >= num_epochs:
    print(f"Checkpoint says next_epoch={start_epoch} but num_epochs={num_epochs}; nothing to run.")

model.train()
for epoch in range(start_epoch, num_epochs):
    # ============ TRAINING ============
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item()
        n_batches += 1

    denom = max(n_batches, 1)
    avg_train_loss = total_loss / denom
    
    # ============ VALIDATION ============
    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            if torch.isfinite(outputs.loss):
                val_loss += outputs.loss.item()
                val_batches += 1
    
    avg_val_loss = val_loss / max(val_batches, 1)
    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Save best adapter when validation improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"  → New best model! Saving adapter to {CHECKPOINT_DIR.resolve()} ...")
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(CHECKPOINT_DIR))
        processor.save_pretrained(str(CHECKPOINT_DIR))

    # Always persist optimizer/scheduler/epoch (resume) even if this epoch was not a new best
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "next_epoch": epoch + 1,
            "best_val_loss": best_val_loss,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "num_epochs": num_epochs,
            "len_train_loader": len(train_loader),
        },
        training_state_path,
    )

    model.train()

Epoch 1: 100%|██████████| 1013/1013 [39:10<00:00,  2.32s/it]


Epoch 1 loss: 4.4421


Epoch 2: 100%|██████████| 1013/1013 [39:06<00:00,  2.32s/it]


Epoch 2 loss: 4.3715


Epoch 3: 100%|██████████| 1013/1013 [38:07<00:00,  2.26s/it]


Epoch 3 loss: nan


Epoch 4: 100%|██████████| 1013/1013 [37:17<00:00,  2.21s/it]


Epoch 4 loss: nan


Epoch 5: 100%|██████████| 1013/1013 [37:17<00:00,  2.21s/it]

Epoch 5 loss: nan
